# Problems
1) Some weird characters in encoder --> filter out in non english?

# Packages

In [27]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence


Import Data

In [28]:
data = pd.read_csv("../../data/processed/trial_processed.csv")
data.head()

,Unnamed: 0,Title,Poem,Poet,Genre
0,0,Search,Wandered tonight through a cityas ruined as a ...,Hester Knibbe,Fear
1,1,A Poem for the Cruel Majority,The cruel majority emerges!Hail to the cruel m...,Jerome Rothenberg,Anger
2,2,“Do Not Embrace Your Mind’s New Negro Friend”,Do not embrace your mind’s new negro friendOr ...,William Meredith,Anger
3,3,The Greatest Love,She is sixty. She livesthe greatest love of he...,Anna Swir,Love
4,4,Bilingual/Bilingüe,"My father liked them separate, one there,one h...",Rhina P. Espaillat,Anger


# Phase 0: Make Tokenizer

In [29]:
corpus = data["Title"].str.cat() + data["Poem"].str.cat()
characters = sorted(list(set(corpus)))



In [30]:
stoi = {s: i for i, s in enumerate(characters)}
itos = {i: s for i, s in enumerate(characters)}

encode = lambda s: [stoi[c] for c in s]
decode = lambda i: "".join([itos[c] for c in i])
decode(encode("hi"))


'hi'

# Phase 1: CSV --> Tensors

In [31]:
### make a list of encoded examples
sequences = []

BOS = "START"
EOS = "END"

for _, row in data.iterrows():
    text = (
            f"{BOS}<title>{row['Title']}</title>"
            f"<poem>{row['Poem']}{EOS}"
        )

    sequences.append(encode(text))

len(sequences)

9119

In [32]:
decode(sequences[0])

'START<title>Search</title><poem>Wandered tonight through a cityas ruined as a body with brokenribs and a bared heart. Looked for you there with cookies in my pocket, searched for a sigh, for movement in demolished streets and alleys. Tonightsince I’d forgotten for a moment where you are,I searched for you with hope in my bones.But no matter how I lured you with my voiceand my eyes, walls of debris grew up steadily around you, cellars seemedto creep around you. I remained alonewith those cookies in my pocketand kept calling and walking.END'

# Phase 2: Dataset

In [33]:
class PoemDataset(Dataset):
    def __init__(self, sequences):
        self.sequences = sequences

    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        return torch.tensor(self.sequences[idx], dtype=torch.long)

In [34]:
dataset = PoemDataset(sequences)


# Phase 3: Collate fn

In [35]:
PAD_ID = 0

In [36]:
def collate_fn(batch):
    x = [seq[:-1] for seq in batch]
    y = [seq[1:] for seq in batch]

    x = pad_sequence(x, batch_first=True, padding_value=PAD_ID)
    y = pad_sequence(y, batch_first=True, padding_value=-100)

    pad_mask = (x == PAD_ID)

    return x, y, pad_mask

# Phase 4: Dataloader

In [ ]:
ds_loader = DataLoader(dataset, batch_size = 3, shuffle=True, collate_fn=collate_fn)

In [38]:
x, y, pad_mask = next(iter(ds_loader))

In [39]:
decode(x[0].tolist())

'START<title>XLIV from The Arab Apocalypse</title><poem>Where do you want ghosts to reside?In our wakeful hours there are flowers which produce nightmaresWe burned continents of silence   the future of nationsthe breathing of the fighters got thicker   became like oxen’sthere is in that breath sparkles of scorched flesh and the fainting of starswe crucify Gilgamesh on a TANK Viking II reaches MarsImam Ali dances over a nuclear blastcursed are the clouds which repel watercursed are the Arabs who fell tall and haggard eucalyptus treesEN\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t\t

In [49]:
x[0]

tensor([ 52,  53,  34,  51,  53,  29,  84,  73,  84,  76,  69,  31,  57,  45,
         42,  55,   1,  70,  82,  79,  77,   1,  53,  72,  69,   1,  34,  82,
         65,  66,   1,  34,  80,  79,  67,  65,  76,  89,  80,  83,  69,  29,
         16,  84,  73,  84,  76,  69,  31,  29,  80,  79,  69,  77,  31,  56,
         72,  69,  82,  69,   1,  68,  79,   1,  89,  79,  85,   1,  87,  65,
         78,  84,   1,  71,  72,  79,  83,  84,  83,   1,  84,  79,   1,  82,
         69,  83,  73,  68,  69,  32,  42,  78,   1,  79,  85,  82,   1,  87,
         65,  75,  69,  70,  85,  76,   1,  72,  79,  85,  82,  83,   1,  84,
         72,  69,  82,  69,   1,  65,  82,  69,   1,  70,  76,  79,  87,  69,
         82,  83,   1,  87,  72,  73,  67,  72,   1,  80,  82,  79,  68,  85,
         67,  69,   1,  78,  73,  71,  72,  84,  77,  65,  82,  69,  83,  56,
         69,   1,  66,  85,  82,  78,  69,  68,   1,  67,  79,  78,  84,  73,
         78,  69,  78,  84,  83,   1,  79,  70,   1,  83,  73,  